<a href="https://colab.research.google.com/github/brittanybee-alt/Updated-Capstone-Project---Brittany-Barrion/blob/main/DC_BB_Capstone_Two_Data_Wrangling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone Two: Data Wrangling
## Where You Grow Up Matters: Predicting Adult Income for Low-Income Children in the DC Region

**Project question:** Which neighborhood characteristics best predict adult income for children raised in low-income families in the DC metro area?

### Data Source
All data comes from **The Opportunity Atlas** (Opportunity Insights and the U.S. Census Bureau):

**[The Opportunity Atlas: household income at age 35 for children raised in low-income families, DC region](https://www.opportunityatlas.org/?%7B%22mapping%22%3A%22kfr%22%2C%22dataToggles%22%3A%7B%22parentIncome%22%3A%22p25%22%2C%22childRace%22%3A%22rP%22%2C%22childGender%22%3A%22gP%22%2C%22cohortTimeline%22%3A%22e%22%7D%2C%22compareDataToggles%22%3A%7B%22parentIncome%22%3A%22pall%22%2C%22childRace%22%3A%22rP%22%2C%22childGender%22%3A%22gP%22%2C%22cohortTimeline%22%3A%22e%22%7D%2C%22selectedChapterId%22%3A%22chapter1%22%2C%22compareFeatures%22%3A%22loc%22%2C%22selectedFeatureByID%22%3A11001010200%2C%22filters%22%3A%5B%5D%2C%22mode%22%3A%22standardMode%22%2C%22mapBounds%22%3A%5B%5B-77.29824387656735%2C38.72686964592006%5D%2C%5B-76.79125493105334%2C39.0687791383437%5D%5D%2C%22floatingPanelIsMinimized%22%3Afalse%2C%22showStoryMarkers%22%3Atrue%2C%22showHistogram%22%3Afalse%2C%22propertyShownOnMap%22%3A%22kfr_rP_gP_p25_e%22%7D)**

The linked map view shows household income in adulthood for children whose parents were at the 25th percentile of the national income distribution (low-income families), for all races and genders (`kfr_rP_gP_p25`). Two Census-tract-level files were downloaded from this map view:

| File | What it contains | Role in the project |
|---|---|---|
| `shown_tract_kfr_rP_gP_p25.csv` | Average household income at age 35 for children from low-income families, by tract | **Target** (what we predict) |
| `shown_tract_allCharacteristics.csv` | Neighborhood characteristics by tract (poverty rate, rent, single-parent share, job density and more) | **Predictors** (features) |

A Census tract is a small neighborhood of roughly 4,000 residents. Both files cover the same DC-area tracts in Washington DC, Maryland and Virginia.

### Steps (following the outline)
1. Data Collection: loading and joining
2. Data Organization: file structure and GitHub
3. Data Definition: column names, data types, descriptions, unique values and ranges
4. Data Cleaning: missing values and duplicates

### Setup

In [5]:
import os
import glob
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

## 2. Data Organization (folder setup)
*The outline lists Data Organization second, but the folders have to exist before the data can be loaded, so they're created here first. The GitHub discussion is in Section 2 below, after loading.*

The project uses a standard data science folder structure:
```
capstone/
├── data/
│   ├── raw/          # original downloads from the Opportunity Atlas, never edited
│   └── processed/    # cleaned files created by these notebooks
├── notebooks/        # one notebook per capstone step
└── figures/          # saved charts for the final report
```
Keeping raw data separate from processed data means the original downloads are never overwritten, so every cleaning step can be traced and repeated.

In [6]:
for folder in ['data/raw', 'data/processed', 'notebooks', 'figures']:
    os.makedirs(folder, exist_ok=True)
print('Folders ready')

Folders ready


**In Google Colab:** upload the two CSV files into the `data/raw` folder (Files panel on the left → right-click `data/raw` → Upload), then run the next cell.

## 1. Data Collection
**Goal:** Organize the data to streamline the next steps of the capstone.

### Data loading
I use `glob` to find each file in `data/raw`, so the notebook still works if the downloaded file name has a suffix such as `-2`. Each file is then read with `pd.read_csv()`.

In [7]:
raw_files = sorted(glob.glob('data/raw/*.csv'))
print('Files in data/raw:')
for f in raw_files:
    print('  ', f)

outcome_path = glob.glob('/content/shown_tract_kfr_rP_gP_p25-3.csv')[0]
chars_path = glob.glob('/content/shown_tract_allCharacteristics-2.csv')[0]

# utf-8-sig removes a hidden byte-order mark at the start of the characteristics file
outcomes = pd.read_csv(outcome_path, encoding='utf-8-sig')
chars = pd.read_csv(chars_path, encoding='utf-8-sig')

print('\nOutcomes file:', outcomes.shape)
print('Characteristics file:', chars.shape)

Files in data/raw:

Outcomes file: (756, 3)
Characteristics file: (760, 15)


In [8]:
outcomes.head()

,tract,Name,Household_Income_at_Age_35_rP_gP_p25
0,51059480203,"Tysons, VA",69199.0
1,24031701002,"Rockville, MD",69034.0
2,51013100400,"Arlington, VA",68322.0
3,51013101601,"North Highland, Arlington, VA",65785.0
4,51510200702,"Eisenhower East, Alexandria, VA",64583.0


In [9]:
chars.head()

,tract,Name,Median_Rent_2012-16,Job_Growth_Rate_from_2004_to_2013,Median_Hhold._Income_of_Residents_in_2012-16,Median_Hhold._Income_of_Residents_in_1990,Poverty_Rate_in_2012-16,Fraction_College_Graduates_in_2012-16,Fraction_Non-White_in_2010,Foreign-Born_Share_in_2012-16,Fraction_Single_Parents_in_2012-16,Population_Density_in_2010,Density_of_Jobs_in_2013,Fraction_with_Short_Work_Commutes_in_2012-16,Census_Response_Rate_Social_Capital_Proxy
0,11001000100,"Georgetown, Washington, DC",2040.0,NaN,165105.0,106198,0.0331,0.8995,0.1407,0.2046,0.1473,6773.0,12139.0,0.1530,81.6
1,11001000201,"Georgetown, Washington, DC",NaN,NaN,NaN,76142,0.6627,0.5397,0.3003,0.1455,NaN,18737.0,25718.0,0.7611,100.0
2,11001000202,"Georgetown, Washington, DC",2228.0,NaN,159367.0,108018,0.2082,0.9459,0.1819,0.1967,0.2252,17500.0,53848.0,0.3000,78.7
3,11001000300,"Northwest Washington, Washington, DC",1880.0,NaN,129493.0,80450,0.1745,0.8894,0.1680,0.1719,0.0970,15820.0,2036.0,0.1941,77.0
4,11001000400,"Naval Observatory, Washington, DC",2282.0,NaN,179146.0,199261,0.1036,0.7926,0.2103,0.2533,0.1077,2458.0,4806.0,0.1605,75.6


### Data joining
Both files have one row per Census tract and share the `tract` ID column, so the right tool is a **merge** on `tract` (a `concat` would stack rows instead of lining up columns for the same tract).

Before merging, I check how well the tract IDs line up.

In [10]:
out_ids = set(outcomes['tract'])
char_ids = set(chars['tract'])
print('Tracts in both files:        ', len(out_ids & char_ids))
print('Only in outcomes file:       ', len(out_ids - char_ids))
print('Only in characteristics file:', len(char_ids - out_ids))

chars[~chars['tract'].isin(outcomes['tract'])][['tract', 'Name', 'Population_Density_in_2010']]

Tracts in both files:         756
Only in outcomes file:        0
Only in characteristics file: 4


,tract,Name,Population_Density_in_2010
540,51013980100,"Arlington, VA",1.058
541,51013980200,"Arlington, VA",2.853
714,51059980100,"Springfield, VA",8.163
715,51059980300,"McLean, VA",4.464


Every tract in the outcomes file has a match. The 4 tracts found only in the characteristics file are special land-use tracts (tract numbers starting with 98) with almost no residents (population density of 1 to 8 people per square mile), such as parkland or military land. They have no income outcome to predict, so an **inner merge** correctly leaves them out.

`validate='one_to_one'` makes pandas raise an error if either file has a repeated tract, which guards against accidentally duplicating rows.

In [11]:
df = outcomes.merge(chars, on='tract', how='inner', validate='one_to_one', suffixes=('', '_chars'))

# Both files carry a 'Name' column; confirm they agree, then keep one copy
print('Rows where the two Name columns differ:', (df['Name'] != df['Name_chars']).sum())
df = df.drop(columns='Name_chars')

print('Merged dataset:', df.shape)
df.head()

Rows where the two Name columns differ: 0
Merged dataset: (756, 16)


,tract,Name,Household_Income_at_Age_35_rP_gP_p25,Median_Rent_2012-16,Job_Growth_Rate_from_2004_to_2013,Median_Hhold._Income_of_Residents_in_2012-16,Median_Hhold._Income_of_Residents_in_1990,Poverty_Rate_in_2012-16,Fraction_College_Graduates_in_2012-16,Fraction_Non-White_in_2010,Foreign-Born_Share_in_2012-16,Fraction_Single_Parents_in_2012-16,Population_Density_in_2010,Density_of_Jobs_in_2013,Fraction_with_Short_Work_Commutes_in_2012-16,Census_Response_Rate_Social_Capital_Proxy
0,51059480203,"Tysons, VA",69199.0,2138.0,0.0479,90366.0,128382,0.0673,0.6614,0.3958,0.6029,0.1672,14264.0,44042.0,0.1916,76.2
1,24031701002,"Rockville, MD",69034.0,3193.0,0.0733,200965.0,169391,0.0262,0.8069,0.2914,0.2444,0.0787,3554.0,1835.0,0.1747,88.3
2,51013100400,"Arlington, VA",68322.0,3501.0,-0.0293,202909.0,162342,0.0286,0.8635,0.1214,0.1138,0.0941,2358.0,120.5,0.2106,91.2
3,51013101601,"North Highland, Arlington, VA",65785.0,1982.0,-0.0358,119122.0,74114,0.1455,0.8462,0.2323,0.1876,0.2963,4899.0,2037.0,0.1230,79.5
4,51510200702,"Eisenhower East, Alexandria, VA",64583.0,2186.0,0.0337,110481.0,64247,0.0222,0.8253,0.3072,0.1373,0.4101,8445.0,13782.0,0.1798,82.1


## 2. Data Organization
**Goal:** Create a file structure and add the work to the project's GitHub repository.

### File structure
The folders were created at the top of this notebook. Listing them with `glob` confirms the layout:

In [12]:
for path in sorted(glob.glob('**/*', recursive=True)):
    if not path.startswith(('sample_data', '.')):
        print(path)

data
data/processed
data/raw
figures
notebooks
shown_tract_allCharacteristics-2.csv
shown_tract_kfr_rP_gP_p25-3.csv


### GitHub
All notebooks, the raw data files and the processed data files are saved to the project's GitHub repository. Using GitHub matters for three reasons:
- **Version control:** every change is saved with a history, so earlier versions can be recovered if something breaks.
- **Reproducibility:** my mentor, or anyone else, can download the repository and re-run each step from the raw data.
- **Portfolio:** the finished capstone becomes public evidence of my work for future employers.

In Colab, each notebook is saved to GitHub with **File → Save a copy in GitHub**. The data files are uploaded to the same repository under `data/raw` and `data/processed`.

## 3. Data Definition
**Goal:** Understand the data features to inform the next steps of the project.

### Column names
**Do the column names correspond to what the columns store?** Mostly yes, but they're long and contain spaces, periods and symbols (for example `Median_Hhold._Income_of_Residents_in_2012-16`), which makes them hard to type in code. I rename each one to a short, clear `snake_case` name. The original names are kept in the data dictionary below.

In [13]:
rename_map = {
    'tract': 'tract',
    'Name': 'name',
    'Household_Income_at_Age_35_rP_gP_p25': 'income_age35',
    'Median_Rent_2012-16': 'median_rent',
    'Job_Growth_Rate_from_2004_to_2013': 'job_growth',
    'Median_Hhold._Income_of_Residents_in_2012-16': 'median_hh_income_2016',
    'Median_Hhold._Income_of_Residents_in_1990': 'median_hh_income_1990',
    'Poverty_Rate_in_2012-16': 'poverty_rate',
    'Fraction_College_Graduates_in_2012-16': 'frac_college_grad',
    'Fraction_Non-White_in_2010': 'frac_nonwhite',
    'Foreign-Born_Share_in_2012-16': 'frac_foreign_born',
    'Fraction_Single_Parents_in_2012-16': 'frac_single_parents',
    'Population_Density_in_2010': 'pop_density',
    'Density_of_Jobs_in_2013': 'job_density',
    'Fraction_with_Short_Work_Commutes_in_2012-16': 'frac_short_commute',
    'Census_Response_Rate_Social_Capital_Proxy': 'census_response_rate',
}
original_names = {new: old for old, new in rename_map.items()}
df = df.rename(columns=rename_map)
df.columns.tolist()

['tract',
 'name',
 'income_age35',
 'median_rent',
 'job_growth',
 'median_hh_income_2016',
 'median_hh_income_1990',
 'poverty_rate',
 'frac_college_grad',
 'frac_nonwhite',
 'frac_foreign_born',
 'frac_single_parents',
 'pop_density',
 'job_density',
 'frac_short_commute',
 'census_response_rate']

### Data types
**Are the data types sensible?**

In [14]:
df.dtypes

,0
tract,int64
name,object
income_age35,float64
median_rent,float64
job_growth,float64
median_hh_income_2016,float64
median_hh_income_1990,int64
poverty_rate,float64
frac_college_grad,float64
frac_nonwhite,float64


Two changes are needed:
- **`tract` is stored as a number, but it's an ID.** Doing math on it (such as averaging) would be meaningless, and the first digits carry meaning: digits 1-2 are the state code and digits 1-5 are the county code. I convert it to text.
- **State and county are hidden inside other columns.** I extract **state** from the end of `name` with `str.extract()`, and **county** from the first 5 digits of `tract`. These give two categorical columns to compare subgroups later in EDA.

In [15]:
df['tract'] = df['tract'].astype(str).str.zfill(11)

df['state'] = df['name'].str.extract(r',\s*([A-Z]{2})$', expand=False)

county_names = {
    '11001': 'District of Columbia',
    '24031': 'Montgomery County, MD',
    '24033': "Prince George's County, MD",
    '24003': 'Anne Arundel County, MD',
    '51013': 'Arlington County, VA',
    '51059': 'Fairfax County, VA',
    '51510': 'Alexandria city, VA',
    '51600': 'Fairfax city, VA',
    '51610': 'Falls Church city, VA',
}
df['county'] = df['tract'].str[:5].map(county_names)

print('Tracts with no state found: ', df['state'].isna().sum())
print('Tracts with no county found:', df['county'].isna().sum())
print()
print(df['state'].value_counts())
print()
print(df['county'].value_counts())

Tracts with no state found:  0
Tracts with no county found: 0

state
MD    304
VA    273
DC    179
Name: count, dtype: int64

county
Prince George's County, MD    187
District of Columbia          179
Fairfax County, VA            172
Montgomery County, MD         115
Arlington County, VA           57
Alexandria city, VA            38
Fairfax city, VA                3
Falls Church city, VA           3
Anne Arundel County, MD         2
Name: count, dtype: int64


In [16]:
# Move the ID and category columns to the front
front = ['tract', 'name', 'state', 'county', 'income_age35']
df = df[front + [c for c in df.columns if c not in front]]
df.dtypes

,0
tract,object
name,object
state,object
county,object
income_age35,float64
median_rent,float64
job_growth,float64
median_hh_income_2016,float64
median_hh_income_1990,int64
poverty_rate,float64


### Description of the columns (data dictionary)

In [17]:
descriptions = {
    'tract': 'Census tract ID (11 digits: state + county + tract)',
    'name': 'Neighborhood or place name of the tract',
    'state': 'State the tract is in (DC, MD or VA)',
    'county': 'County or independent city the tract is in',
    'income_age35': 'TARGET: average household income at age 35 ($) for children whose parents were at the 25th income percentile',
    'median_rent': 'Median gross rent ($/month), 2012-16',
    'job_growth': 'Annual job growth rate, 2004-2013',
    'median_hh_income_2016': 'Median household income of residents ($), 2012-16',
    'median_hh_income_1990': 'Median household income of residents ($), 1990',
    'poverty_rate': 'Share of residents below the poverty line, 2012-16',
    'frac_college_grad': 'Share of adults with a college degree, 2012-16',
    'frac_nonwhite': 'Share of residents who are non-white, 2010',
    'frac_foreign_born': 'Share of residents born outside the U.S., 2012-16',
    'frac_single_parents': 'Share of households with children headed by a single parent, 2012-16',
    'pop_density': 'Residents per square mile, 2010',
    'job_density': 'Jobs per square mile, 2013',
    'frac_short_commute': 'Share of workers with a commute under 15 minutes, 2012-16',
    'census_response_rate': 'Census mail-in response rate (%), a proxy for social capital',
}

numeric = df.select_dtypes('number')
data_dictionary = pd.DataFrame({
    'original_name': [original_names.get(c, '(created in this notebook)') for c in df.columns],
    'description': [descriptions[c] for c in df.columns],
    'dtype': df.dtypes.astype(str).values,
    'missing': df.isna().sum().values,
    'n_unique': df.nunique().values,
    'pct_unique': (df.nunique() / len(df) * 100).round(1).values,
    'min': [numeric[c].min() if c in numeric else np.nan for c in df.columns],
    'max': [numeric[c].max() if c in numeric else np.nan for c in df.columns],
}, index=df.columns)
data_dictionary

,original_name,description,dtype,missing,n_unique,pct_unique,min,max
tract,tract,Census tract ID (11 digits: state + county + t...,object,0,756,100.0,NaN,NaN
name,Name,Neighborhood or place name of the tract,object,0,189,25.0,NaN,NaN
state,(created in this notebook),"State the tract is in (DC, MD or VA)",object,0,3,0.4,NaN,NaN
county,(created in this notebook),County or independent city the tract is in,object,0,9,1.2,NaN,NaN
income_age35,Household_Income_at_Age_35_rP_gP_p25,TARGET: average household income at age 35 ($)...,float64,4,741,98.0,19515.0000,69199.0000
median_rent,Median_Rent_2012-16,"Median gross rent ($/month), 2012-16",float64,15,609,80.6,241.0000,3501.0000
job_growth,Job_Growth_Rate_from_2004_to_2013,"Annual job growth rate, 2004-2013",float64,179,507,67.1,-0.2389,0.3877
median_hh_income_2016,Median_Hhold._Income_of_Residents_in_2012-16,"Median household income of residents ($), 2012-16",float64,3,736,97.4,14509.0000,246886.0000
median_hh_income_1990,Median_Hhold._Income_of_Residents_in_1990,"Median household income of residents ($), 1990",int64,0,637,84.3,11176.0000,272105.0000
poverty_rate,Poverty_Rate_in_2012-16,"Share of residents below the poverty line, 201...",float64,0,634,83.9,0.0000,0.6952


**Note on timing:** the children in this data grew up mostly in the 1980s and 1990s, while most neighborhood characteristics were measured in 2010-2016. Neighborhoods tend to change slowly, so recent measures are a reasonable stand-in, but `median_hh_income_1990` is the measure closest to when these children were growing up. I'll keep this in mind when interpreting results.

### Counts and percents of unique values

In [18]:
unique_summary = pd.DataFrame({
    'n_unique': df.nunique(),
    'pct_unique': (df.nunique() / len(df) * 100).round(1),
}).sort_values('pct_unique', ascending=False)
unique_summary

,n_unique,pct_unique
tract,756,100.0
pop_density,744,98.4
income_age35,741,98.0
median_hh_income_2016,736,97.4
job_density,735,97.2
frac_college_grad,725,95.9
frac_nonwhite,711,94.0
frac_foreign_born,706,93.4
frac_single_parents,689,91.1
median_hh_income_1990,637,84.3


In [19]:
print('Is tract a unique identifier?', df['tract'].is_unique)
print('Is name unique?             ', df['name'].is_unique)

Is tract a unique identifier? True
Is name unique?              False


- **`tract` is 100% unique**, so it's a valid ID for each row.
- **`name` is not unique.** Many tracts share a place name (for example, many tracts are labeled "Northwest Washington"), so `name` describes a location but can't be used as an ID.
- **`state` (3 values) and `county` (9 values)** are low-cardinality categories, suitable for comparing subgroups.
- The numeric features are nearly all unique, as expected for continuous measures.

### Ranges of values and summary statistics

In [20]:
num_cols = df.select_dtypes('number').columns
stats = df[num_cols].describe().T
stats['median'] = df[num_cols].median()
stats['mode'] = df[num_cols].mode().iloc[0]
stats['range'] = stats['max'] - stats['min']
stats[['count', 'mean', 'median', 'mode', 'std', 'min', 'max', 'range']].round(3)

,count,mean,median,mode,std,min,max,range
income_age35,752.0,36668.394,35837.500,22787.000,9726.279,19515.000,69199.000,49684.000
median_rent,741.0,1748.771,1689.000,3501.000,598.058,241.000,3501.000,3260.000
job_growth,577.0,0.009,0.003,0.000,0.070,-0.239,0.388,0.627
median_hh_income_2016,753.0,98262.409,92232.000,246886.000,46808.602,14509.000,246886.000,232377.000
median_hh_income_1990,756.0,87609.521,82386.500,54553.000,37635.513,11176.000,272105.000,260929.000
poverty_rate,756.0,0.106,0.078,0.022,0.100,0.000,0.695,0.695
frac_college_grad,756.0,0.518,0.529,0.871,0.251,0.015,1.000,0.985
frac_nonwhite,756.0,0.604,0.598,0.939,0.290,0.068,0.998,0.930
frac_foreign_born,756.0,0.230,0.203,0.207,0.150,0.000,0.711,0.711
frac_single_parents,753.0,0.330,0.275,0.000,0.239,0.000,1.000,1.000


**What the summary statistics show, and what to investigate:**
- **Target (`income_age35`):** ranges from about \$19,500 to \$69,200. That's more than a 3-to-1 gap in adult income for children who all started in low-income families, depending only on the neighborhood they grew up in. This is the variation the project tries to explain.
- **Shares (`poverty_rate`, `frac_college_grad`, `frac_nonwhite`, etc.)** all fall between 0 and 1, as they should.
- **`census_response_rate`** is on a 0-100 scale, unlike the other shares. That's fine for now, and scaling is handled in pre-processing.
- **`pop_density` and `job_density`** are very skewed: the mean is far above the median, and the maximum is in the hundreds of thousands. These are dense downtown tracts, not errors, but they'll likely need a log transform in EDA.
- **Capped values:** the most common value of `median_rent` is its maximum, \$3,501 (10 tracts), and 6 tracts share the maximum `median_hh_income_2016` of \$246,886. These are **Census top-codes**: the Census reports "\$3,500 or more" rather than exact values for the most expensive areas. They're real high-cost tracts, not errors, so they're kept. The true values are higher than shown.
- **Missing values:** `job_growth` has far more missing values than any other column, and the target has a few gaps. These are handled in Data Cleaning below.

### Checking for impossible values
Before cleaning, I confirm there are no values that can't be real, such as a share above 1 or a negative income.

In [21]:
share_cols = ['poverty_rate', 'frac_college_grad', 'frac_nonwhite', 'frac_foreign_born',
              'frac_single_parents', 'frac_short_commute']
dollar_cols = ['income_age35', 'median_rent', 'median_hh_income_2016', 'median_hh_income_1990']

print('Shares outside 0-1:          ', ((df[share_cols] < 0) | (df[share_cols] > 1)).sum().sum())
print('Negative dollar values:      ', (df[dollar_cols] < 0).sum().sum())
print('Response rate outside 0-100: ', ((df['census_response_rate'] < 0) | (df['census_response_rate'] > 100)).sum())
print('Negative densities:          ', (df[['pop_density', 'job_density']] < 0).sum().sum())

Shares outside 0-1:           0
Negative dollar values:       0
Response rate outside 0-100:  0
Negative densities:           0


No impossible values were found. Extreme but real values, such as very high density downtown, are kept here and examined as outliers in EDA.

## 4. Data Cleaning
**Goal:** Clean up the data to prepare it for the next steps of the project.

### NA or missing values

In [22]:
print('Any missing values at all?', df.isnull().values.any())

missing = pd.DataFrame({'missing': df.isna().sum(), 'pct_missing': (df.isna().mean() * 100).round(1)})
missing[missing['missing'] > 0].sort_values('missing', ascending=False)

Any missing values at all? True


,missing,pct_missing
job_growth,179,23.7
median_rent,15,2.0
income_age35,4,0.5
median_hh_income_2016,3,0.4
frac_single_parents,3,0.4


#### Step 1: `job_growth`. Is the missing data random, or does it follow a pattern?

In [23]:
df.groupby('state')['job_growth'].apply(lambda s: s.isna().mean() * 100).round(1).rename('pct_missing')

,pct_missing
state,
DC,100.0
MD,0.0
VA,0.0


**Decision: drop the `job_growth` column.** It isn't missing at random: it's missing for **100% of DC tracts** and none of the Maryland or Virginia tracts. The options were:
- *Drop the rows:* this would remove every DC tract, the center of a project about the DC region.
- *Fill the gaps:* there's no DC data at all to base a fill on, so every DC value would be invented.
- *Drop the column:* this loses one feature but keeps every tract. This is the best option.

In [24]:
df = df.drop(columns='job_growth')
df.shape

(756, 17)

#### Step 2: missing target (`income_age35`)

In [25]:
df[df['income_age35'].isna()][['tract', 'name', 'state']]

,tract,name,state
752,11001010700,"Downtown, Washington, DC",DC
753,11001006202,"Northwest Washington, Washington, DC",DC
754,11001006804,"Southeast Washington, Washington, DC",DC
755,11001000201,"Georgetown, Washington, DC",DC


**Decision: drop these 4 tracts.** `income_age35` is the target the models will learn to predict. Filling it in would mean making up the answer, so tracts without it can't be used for training or testing. The Opportunity Atlas leaves this value blank when too few children from the tract are in the data to report a reliable estimate. Losing 4 of 756 tracts (0.5%) has very little effect.

In [26]:
df = df.dropna(subset=['income_age35']).reset_index(drop=True)
print(df.shape)

missing = df.isna().sum()
missing[missing > 0]

(752, 17)


,0
median_rent,12
frac_single_parents,1


#### Step 3: the remaining small gaps
After the first two steps, only a few gaps remain: `median_rent` for 12 tracts and `frac_single_parents` for 1 tract.

In [27]:
df[df[['median_rent', 'frac_single_parents']].isna().any(axis=1)][
    ['tract', 'name', 'county', 'median_rent', 'frac_single_parents', 'median_hh_income_2016']]

,tract,name,county,median_rent,frac_single_parents,median_hh_income_2016
32,51059415600,"Alexandria, VA","Fairfax County, VA",NaN,0.0203,210950.0
58,51059482400,"Vienna, VA","Fairfax County, VA",NaN,0.0254,203115.0
95,51059480300,"McLean, VA","Fairfax County, VA",NaN,0.0393,246886.0
123,51059492100,"Fairfax Station, VA","Fairfax County, VA",NaN,0.0721,201885.0
199,51059460900,"Vienna, VA","Fairfax County, VA",NaN,0.0726,172203.0
228,51059470600,"McLean, VA","Fairfax County, VA",NaN,0.0174,189114.0
241,24031704100,"Kensington, MD","Montgomery County, MD",NaN,0.0428,161710.0
265,51059492201,"Fairfax Station, VA","Fairfax County, VA",NaN,0.1508,199740.0
329,51059415500,"Alexandria, VA","Fairfax County, VA",NaN,0.2854,95051.0
336,11001010800,"Northwest Washington, Washington, DC",District of Columbia,1712.0,NaN,31732.0


**Decision: fill each gap with the median of the same county.** These are only 13 values in 752 tracts (under 2%), so dropping the tracts would throw away good data in every other column. I use the **median** rather than the mean because rent is skewed by a few very expensive tracts. I use the **county** median rather than the overall median because rent varies a lot across the region (Fairfax and Arlington are much more expensive than Prince George's), so a county value is a closer estimate for each tract.

In [28]:
for col in ['median_rent', 'frac_single_parents']:
    df[col] = df[col].fillna(df.groupby('county')[col].transform('median'))

print('Any missing values left?', df.isnull().values.any())

Any missing values left? False


### Duplicates

In [29]:
print('Fully duplicated rows: ', df.duplicated().sum())
print('Duplicated tract IDs:  ', df['tract'].duplicated().sum())
print('Is tract still unique? ', df['tract'].is_unique)

Fully duplicated rows:  0
Duplicated tract IDs:   0
Is tract still unique?  True


**No duplicates.** Each tract appears exactly once, which the `validate='one_to_one'` merge already enforced. No rows need to be removed.

### Save the cleaned data
The cleaned file is the starting point for the EDA notebook.

In [30]:
df.to_csv('data/processed/dc_tracts_clean.csv', index=False)
data_dictionary.to_csv('data/processed/data_dictionary.csv')
print('Saved data/processed/dc_tracts_clean.csv:', df.shape)

Saved data/processed/dc_tracts_clean.csv: (752, 17)


In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 752 entries, 0 to 751
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   tract                  752 non-null    object 
 1   name                   752 non-null    object 
 2   state                  752 non-null    object 
 3   county                 752 non-null    object 
 4   income_age35           752 non-null    float64
 5   median_rent            752 non-null    float64
 6   median_hh_income_2016  752 non-null    float64
 7   median_hh_income_1990  752 non-null    int64  
 8   poverty_rate           752 non-null    float64
 9   frac_college_grad      752 non-null    float64
 10  frac_nonwhite          752 non-null    float64
 11  frac_foreign_born      752 non-null    float64
 12  frac_single_parents    752 non-null    float64
 13  pop_density            752 non-null    float64
 14  job_density            752 non-null    float64
 15  frac_s

## Summary
- **Data Collection:** loaded the Opportunity Atlas outcome file (756 tracts) and neighborhood characteristics file (760 tracts) with `read_csv`, and joined them with a one-to-one inner **merge** on `tract`. The 4 unmatched tracts were uninhabited special land-use tracts.
- **Data Organization:** created a `data/raw`, `data/processed`, `notebooks` and `figures` structure. All work is saved to the project's GitHub repository.
- **Data Definition:** renamed the columns to short, clear names, stored `tract` as a text ID, created `state` and `county` columns, and built a data dictionary with descriptions, data types, unique counts and percents, and value ranges. No impossible values were found.
- **Data Cleaning:**
  - Dropped `job_growth`, because it was missing for all DC tracts.
  - Dropped the 4 tracts with no income outcome.
  - Filled 13 small gaps in `median_rent` and `frac_single_parents` with county medians.
  - Found no duplicates.
- **Result:** `data/processed/dc_tracts_clean.csv`, **752 tracts × 17 columns with no missing values** (1 target, 12 neighborhood features, plus the ID, name, state and county columns). This file is ready for Exploratory Data Analysis.